In [ ]:
# import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import re
import string
import joblib


In [ ]:
# Load datasets
fake =(pd.read_csv('/content/fake.csv.zip'))
true =(pd.read_csv('/content/true.csv.zip'))

FileNotFoundError: [Errno 2] No such file or directory: '/content/fake.csv.zip'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# To print fake data
fake.head()

In [ ]:
# To print real data
true.head()

In [ ]:
#label data
fake['class']=0         # 0-------> fake news
true['class']=1          # 1-------> real news

In [ ]:
# Merge datasets
date =pd.concat([fake,true],axis=0)

In [ ]:
# drop irrelevant columns
data = date.drop(['text','subject','date'],axis=1)    # Focusing only on 'title'

In [ ]:
data.reset_index(inplace=True)

In [ ]:
data.drop(['index'],axis=1,inplace=True)

In [ ]:
data.sample(5)

In [ ]:
# cleaning function
def clear_title(title):
  title =title.lower()
  title = re.sub("\n","",title)
  return title
  # Apply cleaning
  data['title'] = data['title'].apply(clear_title)
  return title

In [ ]:
#Vectorization & Training
x = data['title']
y = data['class']
# Split data into 75% training and 25% testing
xtrain,xtest,ytrain,ytest = train_test_split(x,y,test_size=0.25,random_state=42)

In [ ]:
# Convert text titles to numerical data
from numpy import vectorize
vectorize = TfidfVectorizer()
xv_train = vectorize.fit_transform(xtrain)
xv_test = vectorize.transform(xtest)

In [ ]:
# Create and train the Logistic Regression model
model = LogisticRegression()
model.fit(xv_train,ytrain)

In [ ]:
# Evaluation
predict = model.predict(xv_test)
model.score(xv_test,ytest)
print("Model Accuracy:", model.score(xv_test, ytest))

In [ ]:
print(classification_report(ytest,predict))

In [ ]:
# Save the model and vectorizer for the UI
joblib.dump(vectorize,'vectorize.joblib')
joblib.dump(model,'model.joblib')

In [ ]:
%%writefile app.py
import streamlit as st
import joblib
import re

# ==========================================
# 1. LOAD SAVED ASSETS
# ==========================================
# Ensure these files are in your sidebar folder
model = joblib.load('model.joblib')
vectorize = joblib.load('vectorize.joblib')

# ==========================================
# 2. PREPROCESSING FUNCTION (from your code2)
# ==========================================
def clear_title(title):
    """Clean the input title before prediction"""
    title = title.lower()
    title = re.sub("\n", "", title)
    return title

# ==========================================
# 3. UI DESIGN (Streamlit)
# ==========================================
st.set_page_config(page_title="Fake News Detector", )

st.title(" Fake News Detector")
st.markdown("---")
st.write("Enter a news article title below to check whether it is **Fake** or **Real**.")

# User Input Box
user_input = st.text_area("News Title to Analyze:", placeholder="Paste title here...", height=150)

# Prediction Button
if st.button("Check"):
    if user_input.strip() != "":

        cleaned_text = clear_title(user_input)
        vectorized_text = vectorize.transform([cleaned_text])
        prediction = model.predict(vectorized_text)

        # Display Results
        st.subheader("Prediction Result:")
        if prediction == 1:
            st.success(" This news is **REAL**")
        else:
            st.error(" This news is **FAKE**")
    else:
        st.warning(" Please enter some text to analyze.")


In [ ]:
!pip install streamlit
!npm install -g localtunnel

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501